# 02 - Prepare Insert

Transforma os datasets normalizados de contratos em um banco **SQLite**.

A partir dos 4 arquivos:

* `Dados - Contratos Normalizados/contratos_int.csv`
* `Dados - Contratos Normalizados/contratos_comp.csv`
* `Dados - Contratos Normalizados/contratos_mvno.csv`
* `Dados - Contratos Normalizados/contratos_ran_sharing.csv`

este notebook gera:

1. `inserts.sql` &nbsp;&nbsp;&nbsp;&nbsp;&rarr; comandos `INSERT` de todos os registros
2. `database.sql` &nbsp;&rarr; **versao final** (schema `schema.sql` + inserts) que recria o banco do zero
3. `contratos.db` &rarr; banco SQLite ja materializado (para validacao)

O modelo segue o `diagrama_bd.dbml` / `schema.sql`: cada **linha** de um CSV vira
uma **versao de contrato** (`versao_contrato`); linhas com o mesmo `PROCESSO_ANATEL`
(dentro do mesmo tipo) compartilham o mesmo `contrato`.

## Bibliotecas e caminhos

In [2]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DADOS  = Path('Dados - Contratos Normalizados')
SCHEMA_SQL  = Path('schema.sql')
INSERTS_SQL = Path('inserts.sql')
DATABASE_SQL = Path('database.sql')
DATABASE_DB  = Path('contratos.db')

## Leitura dos datasets

In [3]:
df_int  = pd.read_csv(BASE_DADOS / 'contratos_int.csv')
df_comp = pd.read_csv(BASE_DADOS / 'contratos_comp.csv')
df_mvno = pd.read_csv(BASE_DADOS / 'contratos_mvno.csv')
df_ran  = pd.read_csv(BASE_DADOS / 'contratos_ran_sharing.csv')

for nome, df in [('int', df_int), ('comp', df_comp),
                 ('mvno', df_mvno), ('ran', df_ran)]:
    print(f'{nome:5s}: {df.shape}')

int  : (12364, 17)
comp : (9617, 13)
mvno : (341, 15)
ran  : (16, 20)


## Funções auxiliares

Serialização segura para SQL (escapando aspas, tratando nulos e normalizando datas
para o formato ISO `YYYY-MM-DD`, convencao do SQLite).

In [4]:
def is_null(v):
    return v is None or (isinstance(v, float) and pd.isna(v)) or (v is pd.NaT)


def sql_str(v):
    """Texto -> 'texto' com escape de aspas, ou NULL."""
    if is_null(v):
        return 'NULL'
    return "'" + str(v).replace("'", "''") + "'"


def sql_num(v):
    if is_null(v):
        return 'NULL'
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)


def sql_bool(v):
    if is_null(v):
        return 'NULL'
    return '1' if bool(v) else '0'


def sql_date(v):
    """Normaliza para 'YYYY-MM-DD' ou NULL."""
    if is_null(v):
        return 'NULL'
    d = pd.to_datetime(v, errors='coerce')
    return 'NULL' if pd.isna(d) else "'" + d.strftime('%Y-%m-%d') + "'"


def clean(v):
    """NaN/NaT -> None; aplica strip em strings vazias -> None."""
    if is_null(v):
        return None
    if isinstance(v, str):
        v = v.strip()
        return v or None
    return v

## Estruturas de saída

Mantemos dicionários para deduplicar entidades (empresa, servico, contrato) e
listas com as linhas de cada tabela. Os ids inteiros sao gerados sequencialmente.

In [16]:
# dicionarios de deduplicacao
empresas  = {}   # (razao_social, cnpj) -> id_empresa
servicos  = {}   # codigo ANATEL        -> servico_id
contratos = {}   # (tipo, processo)     -> id_contrato

# linhas de cada tabela
rows_empresa = []; rows_servico = []; rows_contrato = []
rows_interconexao = []; rows_compartilhamento = []
rows_ran = []; rows_mvno = []
rows_versao = []; rows_informe = []; rows_acordao = []
rows_despacho = []; rows_participacao = []

seq = {k: 0 for k in ['empresa', 'servico', 'contrato', 'versao',
                      'informe', 'acordao', 'despacho', 'participacao']}


def next_id(k):
    seq[k] += 1
    return seq[k]

### Funcoes de inserção (com deduplicação)

In [17]:
def get_empresa_id(razao, cnpj):
    razao, cnpj = clean(razao), clean(cnpj)
    if razao is None and cnpj is None:
        return None
    key = (razao, cnpj)
    if key not in empresas:
        eid = next_id('empresa')
        empresas[key] = eid
        rows_empresa.append((eid, cnpj, razao))
    return empresas[key]


def get_servico_id(codigo):
    codigo = clean(codigo)
    if codigo is None:
        return None
    codigo = int(codigo)
    if codigo not in servicos:
        servicos[codigo] = codigo           # usa o proprio codigo ANATEL como id
        rows_servico.append((codigo, str(codigo), None))
    return servicos[codigo]


def get_contrato_id(tipo, processo):
    key = (tipo, clean(processo))
    if key not in contratos:
        cid = next_id('contrato')
        contratos[key] = cid
        rows_contrato.append((cid, clean(processo), tipo))
    return contratos[key]


def add_informe(sei, data, texto):
    if clean(sei) is None:
        return None
    iid = next_id('informe')
    rows_informe.append((iid, clean(sei), data, clean(texto)))
    return iid


def add_acordao(sei, data, texto):
    if clean(sei) is None:
        return None
    aid = next_id('acordao')
    rows_acordao.append((aid, clean(sei), data, clean(texto)))
    return aid


def add_despacho(sei, data):
    if clean(sei) is None:
        return None
    did = next_id('despacho')
    rows_despacho.append((did, clean(sei), data))
    return did


def add_versao(cid, num_seq, acordo_tipo, protocolo, conclusao, obs,
               informe_id=None, acordao_id=None, despacho_id=None):
    vid = next_id('versao')
    rows_versao.append((vid, cid, num_seq, clean(acordo_tipo), protocolo,
                        conclusao, clean(obs), informe_id, acordao_id, despacho_id))
    return vid


def add_part(vid, eid, papel, ordem, vigente, servico_id=None, modalidade=None):
    if eid is None:
        return
    pid = next_id('participacao')
    rows_participacao.append((pid, vid, eid, papel, ordem, servico_id,
                              clean(modalidade), vigente))

## 1) Interconexão

Duas prestadoras por versão, cada uma com servico (codigo ANATEL) e modalidade STFC.
O documento associado e o **despacho decisório**.

In [18]:
for _, r in df_int.iterrows():
    cid = get_contrato_id('interconexao', r['PROCESSO_ANATEL'])
    did = add_despacho(r['DESPACHO_DECISORIO_SEI'], r['DESPACHO_DECISORIO_DATA'])
    vid = add_versao(cid, r['NUM_SEQUENCIA'], r['ACORDO_TIPO'],
                     r['PROTOCOLO_DATA'], r['CONCLUSAO_DATA'], r['OBSERVACAO'],
                     despacho_id=did)
    vig = bool(r['FINAL'])
    add_part(vid, get_empresa_id(r['PRESTADORA_1'], r['PRESTADORA_1_CNPJ']),
             'PRESTADORA_1', 1, vig,
             get_servico_id(r['SERVICO_1']), r['MODALIDADE_STFC_1'])
    add_part(vid, get_empresa_id(r['PRESTADORA_2'], r['PRESTADORA_2_CNPJ']),
             'PRESTADORA_2', 2, vig,
             get_servico_id(r['SERVICO_2']), r['MODALIDADE_STFC_2'])

print('contratos apos interconexao:', len(rows_contrato))

contratos apos interconexao: 1735


## 2) Compartilhamento

Detentora (poste/duto/torre) e Solicitante (operadora). Documento: **informe**.

In [8]:
for _, r in df_comp.iterrows():
    cid = get_contrato_id('compartilhamento', r['PROCESSO_ANATEL'])
    iid = add_informe(r['INFORME_SEI'], r['INFORME_DATA'], None)
    vid = add_versao(cid, r['NUM_SEQUENCIA'], r['ACORDO_TIPO'],
                     r['PROTOCOLO_DATA'], r['CONCLUSAO_DATA'], r['OBSERVACAO'],
                     informe_id=iid)
    vig = bool(r['FINAL'])
    add_part(vid, get_empresa_id(r['DETENTORA'], r['DETENTORA_CNPJ']),
             'DETENTORA', 1, vig)
    add_part(vid, get_empresa_id(r['SOLICITANTE'], r['SOLICITANTE_CNPJ']),
             'SOLICITANTE', 2, vig)

print('contratos apos compartilhamento:', len(rows_contrato))

contratos apos compartilhamento: 4818


## 3) MVNO

Prestadora de origem e credenciada. Os atributos especificos do contrato
(`vigencia_data_fim`, `processo_descredenciamento`) são tomados da versão de
maior `NUM_SEQUENCIA` (estado mais recente). Documento: **despacho**.

In [9]:
mvno_attrs = {}  # cid -> (num_seq, vigencia_fim, processo_descred)

for _, r in df_mvno.iterrows():
    cid = get_contrato_id('mvno', r['PROCESSO_ANATEL'])
    did = add_despacho(r['DESPACHO_DECISORIO_SEI'], r['DESPACHO_DECISORIO_DATA'])
    vid = add_versao(cid, r['NUM_SEQUENCIA'], r['ACORDO_TIPO'],
                     r['PROTOCOLO_DATA'], r['CONCLUSAO_DATA'], r['OBSERVACAO'],
                     despacho_id=did)
    vig = bool(r['FINAL'])
    add_part(vid, get_empresa_id(r['PRESTADORA_ORIGEM'], r['PRESTADORA_ORIGEM_CNPJ']),
             'PRESTADORA_ORIGEM', 1, vig)
    add_part(vid, get_empresa_id(r['CREDENCIADA'], r['CREDENCIADA_CNPJ']),
             'CREDENCIADA', 2, vig)

    ns = r['NUM_SEQUENCIA']
    prev = mvno_attrs.get(cid)
    if prev is None or ns >= prev[0]:
        mvno_attrs[cid] = (ns, clean(r['VIGENCIA_DATA_FIM']),
                           clean(r['PROCESSO_DESCREDENCIAMENTO']))

print('contratos apos mvno:', len(rows_contrato))

contratos apos mvno: 5063


## 4) RAN Sharing

Até três prestadoras por versão. Documentos: **informe** e **acordao** (ambos com
texto). A tecnologia do acordo (`ACORDO_TECNOLOGIA`) e guardada no subtipo.

In [10]:
ran_tec = {}  # cid -> tecnologia

for _, r in df_ran.iterrows():
    cid = get_contrato_id('ran_sharing', r['PROCESSO_ANATEL'])
    iid = add_informe(r['INFORME_SEI'], r['INFORME_DATA'], r['INFORME'])
    aid = add_acordao(r['ACORDAO_SEI'], r['ACORDAO_DATA'], r['ACORDAO'])
    vid = add_versao(cid, r['NUM_SEQUENCIA'], r['ACORDO_TIPO'],
                     r['PROTOCOLO_DATA'], None, r['OBSERVACAO'],
                     informe_id=iid, acordao_id=aid)
    vig = bool(r['FINAL'])
    for n in (1, 2, 3):
        add_part(vid, get_empresa_id(r[f'PRESTADORA_{n}'], r[f'PRESTADORA_{n}_CNPJ']),
                 f'PRESTADORA_{n}', n, vig)
    ran_tec.setdefault(cid, clean(r['ACORDO_TECNOLOGIA']))

print('contratos apos ran_sharing:', len(rows_contrato))

contratos apos ran_sharing: 5078


## Tabelas de subtipo (1:1 com `contrato`)

Geradas a partir do dicionário `contratos`, já com os atributos específicos
de `mvno` e `ran_sharing`.

In [11]:
rows_interconexao.clear(); rows_compartilhamento.clear()
rows_mvno.clear(); rows_ran.clear()

for (tipo, _proc), cid in contratos.items():
    if tipo == 'interconexao':
        rows_interconexao.append((cid,))
    elif tipo == 'compartilhamento':
        rows_compartilhamento.append((cid,))
    elif tipo == 'mvno':
        _, vfim, pdesc = mvno_attrs.get(cid, (0, None, None))
        rows_mvno.append((cid, vfim, pdesc))
    elif tipo == 'ran_sharing':
        rows_ran.append((cid, ran_tec.get(cid)))

resumo = {
    'empresa': len(rows_empresa), 'servico': len(rows_servico),
    'contrato': len(rows_contrato), 'interconexao': len(rows_interconexao),
    'compartilhamento': len(rows_compartilhamento), 'mvno': len(rows_mvno),
    'ran_sharing': len(rows_ran), 'versao_contrato': len(rows_versao),
    'informe': len(rows_informe), 'acordao': len(rows_acordao),
    'despacho': len(rows_despacho), 'participacao': len(rows_participacao),
}
pd.Series(resumo, name='qtd_registros').to_frame()

,qtd_registros
empresa,8948
servico,4
contrato,5078
interconexao,1735
compartilhamento,3083
mvno,245
ran_sharing,15
versao_contrato,22338
informe,9386
acordao,16


## Geracao do `inserts.sql`

Os `INSERT` são agrupados em lotes de 500 linhas para manter o arquivo legivel
e o carregamento rápido. Tudo dentro de uma única transação.

In [19]:
def emit(f, tabela, cols, rows, fmts):
    if not rows:
        return
    collist = ', '.join(cols)
    f.write(f'-- {tabela}: {len(rows)} registros\n')
    for i in range(0, len(rows), 500):
        lote = rows[i:i + 500]
        f.write(f'INSERT INTO {tabela} ({collist}) VALUES\n')
        linhas = ['(' + ', '.join(fmt(v) for fmt, v in zip(fmts, row)) + ')'
                  for row in lote]
        f.write(',\n'.join(linhas) + ';\n')
    f.write('\n')


with INSERTS_SQL.open('w', encoding='utf-8') as f:
    f.write('-- Gerado por 02 - Prepare Insert.ipynb\n')
    f.write('PRAGMA foreign_keys = ON;\nBEGIN TRANSACTION;\n\n')

    emit(f, 'empresa', ['id_empresa', 'cnpj', 'razao_social'],
         rows_empresa, [sql_num, sql_str, sql_str])
    emit(f, 'servico', ['servico_id', 'servico_tipo', 'modalidade'],
         rows_servico, [sql_num, sql_str, sql_str])
    emit(f, 'contrato', ['id_contrato', 'id_processo', 'tipo_contrato'],
         rows_contrato, [sql_num, sql_str, sql_str])
    emit(f, 'interconexao', ['id_contrato'], rows_interconexao, [sql_num])
    emit(f, 'compartilhamento', ['id_contrato'], rows_compartilhamento, [sql_num])
    emit(f, 'ran_sharing', ['id_contrato', 'tecnologia'],
         rows_ran, [sql_num, sql_str])
    emit(f, 'mvno', ['id_contrato', 'vigencia_data_fim', 'processo_descredenciamento'],
         rows_mvno, [sql_num, sql_date, sql_str])
    emit(f, 'informe', ['informe_id', 'informe_sei', 'informe_data', 'informe'],
         rows_informe, [sql_num, sql_str, sql_date, sql_str])
    emit(f, 'acordao', ['acordao_id', 'acordao_sei', 'acordao_data', 'acordao'],
         rows_acordao, [sql_num, sql_str, sql_date, sql_str])
    emit(f, 'despacho', ['despacho_id', 'despacho_sei', 'despacho_data'],
         rows_despacho, [sql_num, sql_str, sql_date])
    emit(f, 'versao_contrato',
         ['id_versao', 'id_contrato', 'num_sequencia', 'acordo_tipo',
          'protocolo_data', 'conclusao_data', 'observacao',
          'informe_id', 'acordao_id', 'despacho_id'],
         rows_versao,
         [sql_num, sql_num, sql_num, sql_str, sql_date, sql_date,
          sql_str, sql_num, sql_num, sql_num])
    emit(f, 'participacao',
         ['id_participacao', 'id_versao', 'id_empresa', 'papel',
          'ordem_participacao', 'servico_id', 'modalidade_sftc', 'vigente'],
         rows_participacao,
         [sql_num, sql_num, sql_num, sql_str, sql_num, sql_num, sql_str, sql_bool])

    f.write('COMMIT;\n')

print(f'{INSERTS_SQL} gerado ({INSERTS_SQL.stat().st_size / 1e6:.2f} MB)')

inserts.sql gerado (3.19 MB)


## Versão final: `database.sql`

Concatena o schema (`schema.sql`) com os inserts (`inserts.sql`). Executar este
arquivo em um banco vazio recria todo o SQLite do zero.

In [13]:
schema_txt  = SCHEMA_SQL.read_text(encoding='utf-8')
inserts_txt = INSERTS_SQL.read_text(encoding='utf-8')

with DATABASE_SQL.open('w', encoding='utf-8') as f:
    f.write('-- =====================================================\n')
    f.write('-- Banco de Contratos ANATEL - versao final (SQLite)\n')
    f.write('-- schema.sql + inserts.sql\n')
    f.write('-- =====================================================\n\n')
    f.write(schema_txt)
    f.write('\n\n-- ============ DADOS ============\n\n')
    f.write(inserts_txt)

print(f'{DATABASE_SQL} gerado ({DATABASE_SQL.stat().st_size / 1e6:.2f} MB)')

database.sql gerado (6.45 MB)


## Materialização e validação do banco

Cria `contratos.db` a partir do `database.sql` e confere as contagens e a
integridade referencial (`PRAGMA foreign_key_check`).

In [14]:
if DATABASE_DB.exists():
    DATABASE_DB.unlink()

con = sqlite3.connect(DATABASE_DB)
con.executescript(DATABASE_SQL.read_text(encoding='utf-8'))

cur = con.cursor()
tabelas = ['empresa', 'servico', 'contrato', 'interconexao', 'compartilhamento',
           'ran_sharing', 'mvno', 'versao_contrato', 'informe', 'acordao',
           'despacho', 'participacao']
contagens = {t: cur.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
             for t in tabelas}

violacoes = cur.execute('PRAGMA foreign_key_check').fetchall()
print('Violacoes de chave estrangeira:', len(violacoes))
con.close()

pd.Series(contagens, name='linhas_no_banco').to_frame()

Violacoes de chave estrangeira: 0


,linhas_no_banco
empresa,8948
servico,4
contrato,5078
interconexao,1735
compartilhamento,3083
ran_sharing,15
mvno,245
versao_contrato,22338
informe,9386
acordao,16


### Consulta de exemplo

Contratos com mais versões e suas empresas participantes.

In [15]:
con = sqlite3.connect(DATABASE_DB)
exemplo = pd.read_sql_query("""
    SELECT c.tipo_contrato,
           c.id_processo,
           COUNT(DISTINCT v.id_versao) AS qtd_versoes,
           GROUP_CONCAT(DISTINCT e.razao_social) AS empresas
      FROM contrato c
      JOIN versao_contrato v ON v.id_contrato = c.id_contrato
      JOIN participacao p    ON p.id_versao   = v.id_versao
      JOIN empresa e         ON e.id_empresa  = p.id_empresa
     GROUP BY c.id_contrato
     ORDER BY qtd_versoes DESC
     LIMIT 10
""", con)
con.close()
exemplo

,tipo_contrato,id_processo,qtd_versoes,empresas
0,interconexao,53500.000155/2024-57,960,"TIM S A,R2 DADOS LTDA,CONECTV LTDA,PALMASNET T..."
1,interconexao,53500.015705/2022-71,468,"TIM S A,B R A SERVICOS DE COMUNICACAO LTDA,BNE..."
2,compartilhamento,53500.060352/2021-82,452,"CEMIG DISTRIBUIÇÃO S.A.,CHAVES & LACERDA LTDA,..."
3,interconexao,53500.323929/2022-53,441,"TELEFONICA BRASIL S.A.,55 TELECOM COMUNICACAO ..."
4,interconexao,53500.001117/2023-31,384,"TIM S A,BBS OPTIONS CELULAR LTDA - ME,NÃO INFO..."
5,interconexao,53500.064116/2021-35,312,"TIM S A,JP PROVIDERS EIRELI - EPP,UNION TEL TE..."
6,interconexao,53500.033103/2021-14,192,"TIM S A,NEXT TELECOMUNICAÇÕES DO BRASIL LTDA -..."
7,interconexao,53500.055620/2021-44,180,"TIM S A,UNIVERSO SERVIÇOS DE TELECOMUNICAÇÕES ..."
8,interconexao,53500.019526/2025-55,160,"TIM S A,G2G INTERNET E SERVICOS DE TELECOM LTD..."
9,compartilhamento,53500.000900/2016-58,158,"CELESC DISTRIBUIÇÃO S.A.,W2B,TPA INFORMÁTICA,L..."
